# CCA Developer · M1 — MSO Foundations

This lab covers what happens when a request reaches an LLM, how to pick a model and control its reasoning, how to prompt it, and the two ways to call Claude in production: the **Messages API** and the **Claude Agent SDK**. It uses the AWS Customer Agreement and a long credit agreement as the working examples.

| Section | What you learn | Subtopics |
|---|---|---|
| **[1.1 How LLMs Behave](#s1-1)** | What the model reads, how much fits, and why answers vary | [Tokens](#s1-1-1)<br>[Context window](#s1-1-2)<br>[Sampling and temperature](#s1-1-3)<br>[Non-determinism](#s1-1-4)<br>[Evaluation](#s1-1-5) |
| **[1.2 Model and Reasoning](#s1-2)** | Pick a model for the task and control how much it thinks | [The Claude model family](#s1-2-1)<br>[Alias vs pinned model IDs](#s1-2-2)<br>[Reasoning (thinking)](#s1-2-3) |
| **[1.3 Prompting](#s1-3)** | Use examples to control the output, and keep a conversation going | [Zero-shot](#s1-3-1)<br>[One-shot](#s1-3-2)<br>[Few-shot](#s1-3-3)<br>[Multi-turn](#s1-3-4) |
| **[1.4 The Messages API](#s1-4)** | Choose how your code waits for an answer | [SDK vs raw REST](#s1-4-1)<br>[Synchronous](#s1-4-2)<br>[Streaming](#s1-4-3)<br>[Async](#s1-4-4)<br>[Message Batches](#s1-4-5) |
| **[1.5 Agent SDK vs Messages API](#s1-5)** | Run the same task through both interfaces | [Same task, two interfaces](#s1-5-1)<br>[Which should you use?](#s1-5-2) |

### Before you start
- Have your **Anthropic API key** ready.
- Download both contracts. You upload them in Setup:
  - [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing): short, about 12 pages
  - [PROFRAC HOLDINGS, LLC credit agreement.pdf](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing): long, about 216 pages
- Run the cells from top to bottom. Most cells use a small number of tokens. The cells that send the long contract (1.1.2 and 1.4.5) use the most.

## Setup

Install the tools used in this lab.

| Package | Used for |
|---|---|
| `anthropic` | Calling Claude through the Messages API |
| `PyMuPDF` | Reading text from the PDFs |
| `tiktoken` | Showing how text splits into tokens in 1.1.1 |
| `pyboxen` | Printing answers in readable boxes |

The Claude Agent SDK is installed later, in 1.5, when we need it.

**Install:** install the tools we need.

In [ ]:
!pip install -q -U anthropic PyMuPDF tiktoken pyboxen

Now let's connect to Claude. This cell does three things:
- **Connects to Claude** using your API key. You paste it when asked, and it is not saved. It is stored as `ANTHROPIC_API_KEY`, which both SDKs read, including the Agent SDK in 1.5.
- **Names the models** used in the lab: `HAIKU` and `SONNET`. Change them here and every section follows.
- **Adds `show()`**, a helper that prints text inside a titled box.

**Connect:** ask for your API key, set the model names and define `show()`.

In [ ]:
import os, sys, base64, asyncio
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
import tiktoken
from pyboxen import boxen

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

# Model IDs used throughout the lab. Change them here and every chapter follows.
HAIKU = "claude-haiku-4-5"    # fast + cheap: extraction, classification
SONNET = "claude-sonnet-4-6"  # our default: general analysis


def show(title, text, color="cyan"):
    """Show text inside a titled, colored box."""
    sys.stdout.write(boxen(str(text), title=title, color=color, padding=1) + "\n")

📥 **Upload the contracts:** run the next cell and select **both** PDFs, `AWS1.pdf` and `PROFRAC HOLDINGS, LLC credit agreement.pdf`. If you have not downloaded them yet, use the links in *Before you start*.

In [ ]:
from google.colab import files
uploaded = files.upload()

Now let's turn the PDFs into text, page by page. `contract_text` (the short AWS contract) is used in most of the lab. `long_text` (the credit agreement) shows what happens when a document is too big.

**Extract text:** read each PDF page by page and print how many pages each one has.

In [ ]:
SHORT_PDF = "/content/AWS1.pdf"
LONG_PDF = "/content/PROFRAC HOLDINGS, LLC credit agreement.pdf"


def extract_text(pdf_path):
    """Return the full text of a PDF, with one string per page."""
    with fitz.open(pdf_path) as doc:
        return [page.get_text() for page in doc]


short_pages = extract_text(SHORT_PDF)
long_pages = extract_text(LONG_PDF)

contract_text = "\n".join(short_pages)  # short contract as one string
long_text = "\n".join(long_pages)       # long contract as one string

show("Documents loaded", f"Short contract: {len(short_pages)} pages\nLong contract: {len(long_pages)} pages", "cyan")

## 1.1 How LLMs Behave <a id="s1-1"></a>

Imagine an assistant where a user uploads a contract and asks: *"What is the termination notice period?"* Before adding RAG or agents, understand what happens when that request reaches the model.

| Idea | Question it answers |
|---|---|
| **Tokens** | How is text processed? |
| **Context window** | How much fits in one request? |
| **Sampling** | How is the next word chosen? |
| **Non-determinism** | Why do answers vary? |
| **Evaluation** | How do you test a system like this? |

### 1.1.1 Tokens <a id="s1-1-1"></a>

LLMs do not read characters or words. They read **tokens**: small chunks of text (a word, part of a word, or punctuation), each mapped to a number. Tokens decide **cost, speed and context limits**.

Claude uses a **BPE (Byte Pair Encoding)** tokenizer. Common English words are usually one token. Rare words, code symbols and non-Latin scripts split into several.

| Content | Approximate tokens |
|---|---|
| 1 average English word | ~1.3 |
| 1 line of Python | ~5–12 |
| 1 character of Chinese, Japanese or Korean | ~2–3 |
| 1 JSON object with 5 fields | ~20–40 |
| 1 image (any size) | 1,000–3,000 (depends on detail) |

> Structured output costs more than prose. JSON spends tokens on `{`, `"`, `:` and `}` at every level of nesting, and that adds up at scale.

Now let's tokenize the first page of the contract. We use OpenAI's `cl100k_base` tokenizer only to *show* the idea. Claude has its own tokenizer, but the concept is the same.

**Tokens:** split page 1 into tokens, show where the boundaries fall, and compare characters with tokens.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")  # a public tokenizer, used only to visualise the idea
page = short_pages[0]
ids = enc.encode(page)                      # text -> list of integer token IDs

show("Original text (first 200 chars)", page[:200].replace("\n", " ") + "...", "yellow")
# Decode each ID back to text and join with "|" so you can see where the token boundaries fall
show("First 20 tokens, split with |", "|".join(enc.decode([t]) for t in ids[:20]), "magenta")
show("Token IDs (first 20)", ids[:20], "blue")
show("Size", f"Characters: {len(page):,} | Tokens: {len(ids):,} | ≈ {len(page) / len(ids):.1f} characters per token", "cyan")

**Result:** about 4 characters make 1 token in English (about 1.3 tokens per word). Tokens, not characters or pages, are what you pay for and what limits a request.

Now let's count tokens with Claude's own counter. It is free to call, so use it before sending an expensive request. We compare the same idea written as prose, JSON, code and Chinese.

**Count tokens:** send four versions of the same sentence to the token counter.

In [ ]:
samples = {
    "Prose":   "Either party may terminate this agreement with thirty days written notice.",
    "JSON":    '{"termination": {"by": ["customer", "provider"], "notice_days": 30, "form": "written"}}',
    "Python":  "if notice_days >= 30:\n    terminate(contract)",
    "Chinese": "任何一方均可提前三十天书面通知终止本协议。",
}

rows = []
for label, text in samples.items():
    n = client.messages.count_tokens(
        model=HAIKU, messages=[{"role": "user", "content": text}]
    ).input_tokens  # includes a few tokens of message overhead, so compare the numbers relatively
    rows.append(f"{label:<8} {len(text):>3} characters -> {n:>3} tokens")

show("Characters vs tokens", "\n".join(rows), "cyan")

**Result:** JSON and Chinese need far more tokens per character than plain English. Same information, different bill.

### 1.1.2 Context window <a id="s1-1-2"></a>

The **context window** is the total number of tokens Claude can hold in one request: its working memory. It is **shared**: the contract, your instructions **and** the answer all have to fit.

You set `max_tokens`, the most output the model may write. Input and output together must fit:

```text
input_tokens + max_tokens  ≤  context_window_size
```

For example, 180K tokens of input plus `max_tokens=32768` needs a window of at least about 213K tokens.

**KV cache:** the model keeps the token representations it has already computed, so earlier tokens in a conversation are not recomputed. The cache is bounded by the context window. Once a conversation outgrows it, *you* decide what to drop.

Now let's measure both contracts with the token counter.

**Measure:** count the tokens in the short and the long contract.

In [ ]:
token_counts = {}  # saved so the next cell can reuse the numbers
rows = []

for name, text in [("Short contract", contract_text), ("Long contract", long_text)]:
    n = client.messages.count_tokens(
        model=HAIKU, messages=[{"role": "user", "content": text}]
    ).input_tokens
    token_counts[name] = n
    rows.append(f"{name}: {n:,} tokens")

show("Token counts", "\n".join(rows), "cyan")

Now let's apply the rule: does `input_tokens + max_tokens` fit in the window? Claude Haiku 4.5 has a 200K-token window. Check the docs for the model you use.

**Fit check:** add the planned output budget to each contract's tokens and compare with the window.

In [ ]:
CONTEXT_WINDOW = 200_000  # tokens (Haiku 4.5). Larger windows are available on some models.
MAX_TOKENS = 1000         # the output budget we plan to request

rows = []
for name, n_input in token_counts.items():
    needed = n_input + MAX_TOKENS
    verdict = "✅ fits" if needed <= CONTEXT_WINDOW else "❌ too big"
    rows.append(f"{name}: {n_input:,} input + {MAX_TOKENS:,} output = {needed:,} of {CONTEXT_WINDOW:,} -> {verdict}")

show("Context window check", "\n".join(rows), "cyan")

#### Fill order and prompt caching

Because of the KV cache, **the order of your prompt matters**. Put stable content first and changing content last:

| # | Part | Changes |
|---|---|---|
| 1 | **System prompt** | Rarely: cached first |
| 2 | **Reference documents**, such as the contract | Rarely |
| 3 | **Conversation history** | Grows, oldest turns first |
| 4 | **Current user message** | Every request: at the very end |

Stable content at the front is cached and reused. Changing content at the end does not invalidate it.

> ⚠️ Do not put changing content (timestamps, user IDs) at the start of the system prompt. It changes the front of the prompt on every request, invalidates the cache, and throws away the savings.

Now let's see caching work. The instructions and the contract are the stable prefix, marked with `cache_control`. Only the question changes.

**Caching:** ask two questions on the same contract and print how many tokens were written to and read from the cache.

In [ ]:
def ask_cached(question):
    r = client.messages.create(
        model=SONNET,
        max_tokens=300,
        system=[
            # 1) stable instructions
            {"type": "text", "text": "You are a contract analyst. Answer only from the contract."},
            # 2) stable reference document - cache_control caches everything up to and including this block
            {"type": "text", "text": f"Contract:\n{contract_text}", "cache_control": {"type": "ephemeral"}},
        ],
        # 3) dynamic content goes LAST so it never invalidates the cached prefix
        messages=[{"role": "user", "content": question}],
    )
    u = r.usage
    show("Cache usage", f"cache written: {u.cache_creation_input_tokens or 0:>6,} | cache read: {u.cache_read_input_tokens or 0:>6,} | uncached input: {u.input_tokens:>6,}", "blue")
    return r.content[0].text


show("Call 1 · writes the cache", ask_cached("Who are the parties?"), "yellow")
show("Call 2 · reads the cache", ask_cached("What is the governing law?"), "green")
# If both 'cache written' and 'cache read' are 0, the prefix is shorter than the model's minimum cacheable size.

**Result:** call 1 shows tokens under *cache written*. Call 2 shows about the same number under *cache read*, which costs a fraction of the normal rate. If both are 0, the prefix is shorter than the model's minimum cacheable size.

Tokens are not the only limit. The API also limits **PDF uploads** to 100 pages per request. Now let's send each PDF to Claude directly as a document.

**PDF limit:** send both PDFs as document blocks and ask for a summary of each.

In [ ]:
def pdf_block(path):
    """Wrap a PDF file as a base64 'document' block that Claude can read."""
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode()
    return {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": data}}


for name, path in [("Short (≈12 pages)", SHORT_PDF), ("Long (≈216 pages)", LONG_PDF)]:
    try:
        r = client.messages.create(
            model=HAIKU,
            max_tokens=1000,
            messages=[{"role": "user", "content": [
                pdf_block(path),
                {"type": "text", "text": "Summarize the key terms of this contract."},
            ]}],
        )
        show(f"✅ {name}", r.content[0].text, "green")
    except anthropic.APIError as e:
        show(f"❌ {name} — rejected", e.message, "red")

**Result:** the short PDF is summarized and the long one is rejected. Every request has limits, on tokens *and* on PDF pages. That is why techniques like **chunking and RAG** exist. The rest of this lab sends the contract as plain text.

### 1.1.3 Sampling and temperature <a id="s1-1-3"></a>

An LLM writes **one token at a time**. At each step it gives every possible next token a probability, then **samples** one:

```text
Next token after "Notice period is ..."
"30"     60%     "thirty" 20%     "90" 10%     other 10%
```

**`temperature`** (0–1) controls that choice:
- **Low (0):** almost always picks the most likely token, so answers are consistent.
- **High (1):** more willing to pick less likely tokens, so answers vary.

| Parameter | What it controls | Practical use |
|---|---|---|
| `temperature` | How wide the probability spread is (0–1) | Low (0–0.3) for facts and extraction. High (0.7–1) for creative work |
| `top_p` | Nucleus sampling: a cumulative probability cutoff | Tune **either** temperature **or** `top_p`, not both |
| `top_k` | A hard cap on how many candidate tokens are considered | Rarely tuned. Stick with temperature |
| `max_tokens` | The longest the output may be | Always set it explicitly |

Now let's compare. We ask the same question 3 times at temperature 0 and 3 times at temperature 1.

**Temperature:** run one question three times at each setting and compare the answers.

In [ ]:
question = (
    "In one sentence, describe how either party can terminate this contract.\n\n" + contract_text
)

for temp in (0, 1):          # compare the most consistent setting with the most varied one
    for run in (1, 2, 3):    # same prompt, three runs: do the answers change?
        r = client.messages.create(
            model=HAIKU,
            max_tokens=150,                      # set the output budget explicitly
            extra_body={"temperature": temp},    # sampling control under test
            messages=[{"role": "user", "content": question}],
        )
        show(f"temperature={temp} · run {run}", r.content[0].text, "blue" if temp == 0 else "red")

**Result:** at temperature 0 the three answers are nearly the same. At temperature 1 the wording changes from run to run.

### 1.1.4 Non-determinism <a id="s1-1-4"></a>

At temperature 1 the same input gives different wording. Even at temperature 0, outputs are only **near**-deterministic:

```text
Run 1 → "The notice period is 30 days."
Run 2 → "Either party must give 30 days' notice."
```

Different words, **same meaning**. Even at `temperature=0`, outputs can differ because of:

| Cause | Why |
|---|---|
| **Hardware** | Floating-point differences between servers |
| **Model updates** | A new model version changes behavior |
| **Token-budget pressure** | A model close to its `max_tokens` limit may shorten its answer |
| **Prompt caching** | Small timing differences |

So you cannot test an LLM app by checking for an exact string. Design around it:
- Check **properties** of the output in your eval, not exact strings.
- Use **structured output** (a JSON schema or tool use) to fix the format.
- Test **several samples** to catch edge cases.
- **Pin the model ID** when you need stable behavior (1.2.2).

### 1.1.5 Evaluation <a id="s1-1-5"></a>

Traditional tests use **exact match**: `expected == actual`. With an LLM, many different answers can be correct, so judge the answer on:

| Criterion | Question |
|---|---|
| **Correctness** | Is the extracted term right? |
| **Relevance** | Does it answer the question asked? |
| **Completeness** | Is anything important missing? |
| **Grounding** | Is it supported by the contract text? |

> Do not ask *"Did it produce exactly this sentence?"* Ask *"Is the answer correct?"*

## 1.2 Model and Reasoning <a id="s1-2"></a>

Not every task needs the same intelligence. Extracting a date is simple, summarizing a clause is moderate, and judging whether a termination clause is risky is complex. Pick the model and the amount of thinking to match.

### 1.2.1 The Claude model family <a id="s1-2-1"></a>

There is no "best" model, only trade-offs between **capability, latency and cost**.

| Model | Use case | Strengths | When *not* to use |
|---|---|---|---|
| **Claude Haiku 4.5** | High-volume, latency-sensitive tasks | Fastest and cheapest. Good at extraction and classification | Complex reasoning, long tasks |
| **Claude Sonnet 5** | General purpose (the default) | Best balance of capability, speed and cost | When it has already failed your eval |
| **Claude Opus 5** | Complex research, long reasoning | Highest capability, extended thinking | High volume (too costly) |
| **Claude Fable 5** | Narrative, roleplay, creative writing | Engaging long-form fiction | Factual Q&A, structured extraction |

**Model selection rule:**
1. **Start with Sonnet.**
2. Move **up** to Opus only if Sonnet fails your eval on your task.
3. Move **down** to Haiku only if Sonnet passes but cost or latency is over budget.

> ⚠️ *"This task sounds complex"* is not a reason to pick Opus. Run the eval on your real task first. Sonnet handles most production work.

In this lab the `HAIKU` and `SONNET` variables from Setup pick the model, so you can swap in a newer one in one place.

### 1.2.2 Alias vs pinned model IDs <a id="s1-2-2"></a>

A model is named by an ID string, and there are two kinds:

| Kind | Example | Behavior | Use for |
|---|---|---|---|
| **Alias** | `claude-sonnet-4-6` | Always points to the latest stable release of that model | Development |
| **Pinned** | The same name plus a release date | Points to one specific release. No surprise changes | Production, once you have a validated eval suite |

Now let's ask the API which model IDs exist.

**List models:** print the model IDs the API offers and the two this lab uses.

In [ ]:
from itertools import islice

# Each model is listed with its ID. Dated IDs are pinned releases; names without a date are aliases.
rows = [f"{m.id:<42} {m.display_name}" for m in islice(client.models.list(limit=20), 15)]

show("Available models", "\n".join(rows), "cyan")
show("In this lab", f"{HAIKU} and {SONNET}", "green")

**Result:** IDs with a date are pinned releases. Names without a date are aliases.

### 1.2.3 Reasoning (thinking) <a id="s1-2-3"></a>

Claude has two reasoning modes:
- **Basic (default):** a direct answer. Reasoning happens while it writes. Fast, and enough for most tasks.
- **Extended thinking:** Claude first writes a scratchpad of reasoning (a **thinking block**), then the final answer. It clearly improves accuracy on multi-step problems.

| Task type | Extended thinking needed? |
|---|---|
| Simple Q&A, summaries, extraction | No |
| Code for standard patterns | No |
| Multi-step math or logic | Yes |
| Complex architecture trade-offs | Usually yes |
| Hard competitive-programming problems | Yes |

> **Cost:** thinking tokens count as **output tokens**. `budget_tokens` is a *ceiling*: the model uses what it needs, up to that limit. Turn thinking on only after basic mode has failed your eval.

You control it with the `thinking` parameter:

| Mode | Code | Behavior |
|---|---|---|
| **Off** | `thinking={"type": "disabled"}` | Answers directly |
| **Fixed budget** | `thinking={"type": "enabled", "budget_tokens": 5000}` | Thinks up to a budget you set |
| **Adaptive** | `thinking={"type": "adaptive"}` | Claude decides *whether and how much* to think |

`output_config={"effort": "high"}` sets how hard Claude tries.

> Defaults differ by model: Sonnet 4.6 has thinking off. Sonnet 5 and Opus 5 use adaptive thinking.

Now let's ask a multi-part risk question with adaptive thinking. The response holds separate **thinking** and **text** blocks, so we print both.

**Adaptive thinking:** ask a 5-part risk question and print the thinking block and the answer.

In [ ]:
response = client.messages.create(
    model=SONNET,
    max_tokens=4000,
    thinking={"type": "adaptive"},         # let Claude decide whether (and how much) to think
    output_config={"effort": "high"},      # how hard it should try
    messages=[{"role": "user", "content": f"""
You are a contract risk analyst. For the AWS Customer Agreement below:
1. What termination rights does each party have?
2. What notice periods apply?
3. What happens to the customer's access and content after termination?
4. Is there significant risk for the customer?
5. Give the reviewer a concise recommendation.
Cite section numbers.

Contract:
{contract_text}
"""}],
)

# The response is a list of typed blocks: "thinking" (the scratchpad) and "text" (the final answer)
for block in response.content:
    if block.type == "thinking":
        show("🧠 Thinking", block.thinking, "purple")
    elif block.type == "text":
        show("💬 Answer", block.text, "green")

**Result:** if Claude chose to think, a thinking block comes first, then the answer with section numbers.

Now let's measure what thinking costs. We ask the same question with thinking **off** and with a **fixed budget**, then compare the output tokens.

**Thinking cost:** run one question with thinking off and on, and compare the blocks and output tokens.

In [ ]:
risk_question = f"Is the termination clause risky for the customer? Answer in 3 sentences.\n\nContract:\n{contract_text}"

modes = [
    ("Thinking off", {"type": "disabled"}),
    ("Extended thinking (budget 4000)", {"type": "enabled", "budget_tokens": 4000}),  # ceiling, not a target
]

for label, thinking in modes:
    r = client.messages.create(
        model=SONNET,
        max_tokens=6000,   # must be larger than budget_tokens: thinking + answer share this budget
        thinking=thinking,
        messages=[{"role": "user", "content": risk_question}],
    )
    kinds = [b.type for b in r.content]
    answer = next(b.text for b in r.content if b.type == "text")
    show(label, f"blocks = {kinds} | output tokens = {r.usage.output_tokens}\n\n{answer}", "green")

**Result:** with thinking on, the response has an extra thinking block and more output tokens. Those extra tokens are what thinking costs.

## 1.3 Prompting <a id="s1-3"></a>

**Prompting** means giving the model instructions, context and, optionally, examples. The number of examples is what changes:

| Style | Examples | Use when |
|---|:-:|---|
| **Zero-shot** | 0 | The task is simple and clear on its own |
| **One-shot** | 1 | You want to show the exact output format |
| **Few-shot** | 2–4 | There are different cases or edge cases, such as missing values |

We extract the same key terms three ways. First, a small helper so we do not repeat the API call.

**Helper:** `ask()` sends one prompt and returns Claude's text answer.

In [ ]:
def ask(prompt, model=HAIKU, max_tokens=1000, **kwargs):
    """Send one user prompt and return Claude's text answer."""
    r = client.messages.create(
        model=model, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}], **kwargs,
    )
    return r.content[0].text

### 1.3.1 Zero-shot <a id="s1-3-1"></a>

Instructions only, no examples.

**Zero-shot:** ask for seven key terms with no example of the format.

In [ ]:
zero_shot = f"""
Extract these key terms from the contract: parties, effective date, expiration date,
contract value, payment terms, termination terms, governing law.

Contract:
{contract_text}
"""

show("🎯 Zero-shot", ask(zero_shot))

### 1.3.2 One-shot <a id="s1-3-2"></a>

One example fixes the format.

**One-shot:** show one contract and its JSON, then ask for the same JSON for our contract.

In [ ]:
one_shot = f"""
Extract the key terms as JSON, following the example. Return only JSON.

Example
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term starting January 1, 2025, for $50,000,
payable net 30. Either party may terminate with 30 days' notice. Governed by New York law."
Output:
{{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
  "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": "30 days' notice by either party",
  "governing_law": "New York"}}

Contract:
{contract_text}
Output:
"""

show("🥇 One-shot", ask(one_shot))

### 1.3.3 Few-shot <a id="s1-3-3"></a>

Several examples cover the edge cases. Here the edge case is a missing value, which must become `null`.

**Few-shot:** show three examples, two with missing terms, then ask for the JSON for our contract.

In [ ]:
few_shot = f"""
Extract the key terms as JSON. Use null when a term is not stated. Return only JSON.

Example 1
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term from January 1, 2025 for $50,000, net 30."
Output: {{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
          "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": null, "governing_law": null}}

Example 2
Contract: "This agreement is effective March 3, 2024 between Acme Inc and Beta LLC. Either party may
terminate on 60 days' notice. Delaware law governs."
Output: {{"parties": ["Acme Inc", "Beta LLC"], "effective_date": "2024-03-03", "expiration_date": null,
          "contract_value": null, "payment_terms": null, "termination_terms": "60 days' notice by either party",
          "governing_law": "Delaware"}}

Example 3
Contract: "Services are provided by Orion Corp to Nova Ltd. Fees are USD 8,000 per month."
Output: {{"parties": ["Orion Corp", "Nova Ltd"], "effective_date": null, "expiration_date": null,
          "contract_value": null, "payment_terms": "USD 8,000 per month", "termination_terms": null, "governing_law": null}}

Contract:
{contract_text}
Output:
"""

show("🏅 Few-shot", ask(few_shot))

**Result:**

| Style | What we saw |
|---|---|
| **Zero-shot** | The content is right, but the format is up to the model |
| **One-shot** | The output follows our JSON structure |
| **Few-shot** | Missing values are also handled the same way every time (`null`) |

**Guidelines for good examples:**
- **Quality over quantity:** three excellent, varied examples beat ten average ones.
- **Cover the hard cases:** easy examples add nothing. Edge cases teach the boundary.
- **Match the real format:** if production input has typos, put a typo in an example.
- **Keep examples next to** the instruction they illustrate.

> More examples is not always better. Ask: **"What is the smallest amount of guidance that gives the quality we need?"** If prompts keep growing, consider a stronger model, more reasoning effort or tighter output constraints.

### 1.3.4 Multi-turn <a id="s1-3-4"></a>

The Messages API is **stateless**: Claude only knows what is in the `messages` list you send. For a conversation, you send the whole earlier exchange every time. Be deliberate about how much history you keep. The next lab covers context management.

Now let's hold a two-turn conversation. The follow-up only makes sense with the first turn in the history.

**Multi-turn:** ask who the parties are, keep the reply in the history, then ask a follow-up.

In [ ]:
# Turn 1: a question about the contract
history = [{"role": "user", "content": f"Contract:\n{contract_text}\n\nWho are the parties?"}]
first = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
show("Turn 1", first.content[0].text, "yellow")

# Keep Claude's reply in the history, then add a follow-up that only makes sense with the earlier turns
history.append({"role": "assistant", "content": first.content[0].text})
history.append({"role": "user", "content": "And which law governs it?"})

second = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
show("Turn 2 (follow-up)", second.content[0].text, "green")
show("History", f"Messages sent on the 2nd call: {len(history)}", "cyan")  # history grows every turn -> so does the token bill

**Result:** turn 2 answers the follow-up because the history was sent again. The 2nd call already sends 3 messages, and the count (and the bill) grows every turn.

## 1.4 The Messages API <a id="s1-4"></a>

Every call so far used the **Messages API**: you send messages and get a response. What changes is *how* your code waits for it:

| Pattern | Idea | Use when | What the user sees |
|---|---|---|---|
| **Synchronous** | Send, wait, get the full answer | Internal pipelines, short outputs | The full wait. Not for interactive screens |
| **Streaming** | Receive the answer piece by piece | Any user-facing output | First text in about half a second, then more as it is written |
| **Async** | Send many requests at once | You need several answers fast | Seconds, all together |
| **Batch** | Submit a background job, collect results later | Non-urgent bulk work: nightly reports, large document sets, evals | Within 24 hours, at **50% lower cost** |

```text
Sync:   A ──wait──▶ resp A ; B ──wait──▶ resp B        (one after another)
Async:  A ─┐
        B ─┼──▶ Claude ──▶ responses                   (all at once)
        C ─┘
```

### 1.4.1 SDK vs raw REST <a id="s1-4-1"></a>

Every call is an HTTPS request to `/v1/messages`. The official Python and TypeScript **SDK** wraps it and adds:

| SDK feature | What it saves you |
|---|---|
| **Automatic retries** with exponential backoff | Handling rate-limit errors yourself |
| **Streaming parsing** | Reading server-sent events by hand |
| **Typed** request and response objects | Guessing field names |
| **Helpers** such as token counting | Writing them yourself |

Use the SDK for all production integrations. Use raw REST only when the SDK is not available, for example in an unusual runtime.

Now let's send the same request both ways.

**REST vs SDK:** send one request with raw `requests`, then the same request with the SDK.

In [ ]:
import requests

payload = {
    "model": HAIKU,
    "max_tokens": 100,
    "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
}

# Raw REST: you build the headers, parse the JSON and handle errors / retries yourself
raw = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={
        "x-api-key": os.environ["ANTHROPIC_API_KEY"],
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    },
    json=payload,
    timeout=60,
)
raw.raise_for_status()
show("Raw REST", raw.json()["content"][0]["text"], "yellow")

# SDK: same call, but typed objects, and max_retries gives backoff on rate-limit / server errors
sdk_client = anthropic.Anthropic(max_retries=5)
r = sdk_client.messages.create(**payload)
show("SDK", r.content[0].text, "green")

**Result:** both return the same kind of answer. The SDK version needed no headers or JSON parsing, and `max_retries` adds backoff on rate-limit and server errors.

### 1.4.2 Synchronous <a id="s1-4-2"></a>

The call blocks until the full answer arrives. It is the simplest to write and fine for internal pipelines and scripts. In a user-facing screen the wait feels slow, so stream there instead.

**Synchronous:** ask for five key terms and wait for the whole answer.

In [ ]:
prompt = f"""
Identify the parties, effective date, expiration date, contract value and termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(    # blocks here until the whole answer is ready
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}],
)
show("1️⃣ Synchronous", response.content[0].text)

**Result:** nothing prints until the whole answer is ready.

### 1.4.3 Streaming <a id="s1-4-3"></a>

Text arrives as it is generated. For any user-facing output, stream: the first words appear quickly and the user reads while the rest is written. `client.messages.stream(...)` gives the pieces through `stream.text_stream`.

**Streaming:** stream a summary of the termination rights and collect the pieces.

In [ ]:
with client.messages.stream(       # the SDK parses the server-sent events for us
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": "Summarize the termination rights and notice periods.\n\n" + contract_text}],
) as stream:
    chunks = []
    for chunk in stream.text_stream:          # each chunk is a few tokens of text
        chunks.append(chunk)                  # a box can't be drawn incrementally, so collect the chunks

show("Streamed response", "".join(chunks), "green")

**Result:** the text arrived in small pieces as it was written. A box cannot be drawn piece by piece, so we collect them and print one box at the end. A real screen shows each piece as it arrives.

### 1.4.4 Async <a id="s1-4-4"></a>

`AsyncAnthropic` lets Python do other work while a request is in flight. Define the function with `async def` and wait for the result with `await`. Colab allows `await` directly in a cell.

**Async:** define `ask_async()` and use it for one request.

In [ ]:
async_client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY


async def ask_async(prompt, max_tokens=500):
    r = await async_client.messages.create(
        model=SONNET, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text


show("3️⃣ Async", await ask_async("Summarize the termination rights and risks.\n\n" + contract_text))

#### Many calls in parallel

`asyncio.gather` runs all five requests **at the same time**, so the total time is about the slowest request, not the sum of all five.

**Parallel:** ask five questions at once with `asyncio.gather`.

In [ ]:
topics = ["contract parties", "contract value", "effective date", "termination terms", "governing law"]

answers = await asyncio.gather(
    *(ask_async(f"Identify the {t}.\n\nContract:\n{contract_text}") for t in topics)
)

for topic, answer in zip(topics, answers):
    show(f"4️⃣ {topic}", answer)

**Result:** all five answers arrive in about the time of the slowest one.

### 1.4.5 Message Batches <a id="s1-4-5"></a>

Say we want the **same 3 questions** answered for **both contracts**: 2 × 3 = **6 independent requests**. Instead of sending them one by one, submit them as one **batch** and check back later. Batches finish within **24 hours** and cost **50% less**, so they suit non-urgent bulk work such as nightly reports, large document sets or evals.

```text
2 contracts × 3 questions ──▶ Batch ──▶ check status ──▶ fetch results
```

> 💡 For high-volume overnight jobs, the cheapest combination is **Haiku + the Batches API**: the cheapest model, the batch discount, and no latency requirement.

Now let's submit the batch. Each request gets a `custom_id` so we can match its result later.

**Submit:** build the 6 requests and send them as one batch.

In [ ]:
questions = [
    "What is the total contract value?",
    "What are the payment terms?",
    "What are the termination conditions?",
]

batch_requests = [
    {
        "custom_id": f"{name}_q{i}",  # lets us match each result to its request
        "params": {
            "model": SONNET,
            "max_tokens": 500,
            "messages": [{"role": "user", "content":
                f"Contract:\n{text}\n\nQuestion: {q}\nAnswer only using information from the contract."}],
        },
    }
    for name, text in [("12_page", contract_text), ("216_page", long_text)]
    for i, q in enumerate(questions, 1)
]

batch = client.messages.batches.create(requests=batch_requests)
show("Batch submitted", f"{len(batch_requests)} requests · batch id: {batch.id}", "yellow")

Batches run in the background and can take a few minutes. Now let's wait for this one to finish, then read the results.

**Collect:** check the status every 5 seconds, then print each result with its `custom_id`.

In [ ]:
import time

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    show("Batch status", status.processing_status, "yellow")
    time.sleep(5)

for item in client.messages.batches.results(batch.id):
    if item.result.type == "succeeded":
        show(item.custom_id, item.result.message.content[0].text, "green")
    else:
        show(item.custom_id, f"Request {item.result.type}", "red")

**Result:** each answer is matched to its request by `custom_id`, not by position.

#### Sync vs streaming vs async vs batch

| | Question you are asking | Result arrives |
|---|---|---|
| **Sync** | "I need this answer now, in a script." | After the full wait |
| **Streaming** | "A user is watching." | Piece by piece, first text in about 0.5 s |
| **Async** | "I need several answers, fast." | All together, in seconds |
| **Batch** | "I have many independent jobs and can wait." | Later, in the background (within 24 h, at 50% lower cost) |

## 1.5 Agent SDK vs Messages API <a id="s1-5"></a>

So far **we** wrote the prompt, called `client.messages.create(...)` and got one response back. That is the **Messages API**: low level, one request in, one response out.

The **Claude Agent SDK** works differently. You give Claude a *task*, and it can loop over several turns, use tools (files, bash, search, your own tools) and stream back a sequence of typed messages.

| | Messages API | Claude Agent SDK |
|---|---|---|
| **Call** | `client.messages.create()` | `query()` |
| **Shape** | 1 request → 1 response | 1 task → a stream of messages, many turns |
| **Tools** | You run the tool loop yourself | Built-in loop and tools |
| **Best for** | Single-shot generation and extraction | Multi-step, tool-using workflows |

The SDK runs on the Claude Code CLI, which needs Node.js (already in Colab). It uses the `ANTHROPIC_API_KEY` set in Setup.

**Install:** install the Claude Code CLI and the Agent SDK.

In [ ]:
!npm install -g @anthropic-ai/claude-code
!pip install -q -U claude-agent-sdk

### 1.5.1 Same task, two interfaces <a id="s1-5-1"></a>

Now let's run one task both ways. First the **Messages API**: one call, one response object.

**Messages API:** ask for four key terms with one `messages.create` call.

In [ ]:
task_prompt = f"""
Analyze this contract and identify: 1. parties, 2. effective date, 3. contract value, 4. termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(
    model=SONNET, max_tokens=1000,
    messages=[{"role": "user", "content": task_prompt}],
)
show("📨 Messages API", response.content[0].text)  # the answer is right here

Now the **Claude Agent SDK**. `query()` is async and yields a **stream** of messages. Claude's answer sits inside `AssistantMessage` → `TextBlock`, so we pick it out of the stream. The same loop would carry tool calls if Claude decided to use tools.

**Agent SDK:** run the same task with `query()` and collect the text blocks.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock


async def ask_agent(prompt):
    options = ClaudeAgentOptions(model=SONNET, max_turns=1)  # single turn, for a fair comparison
    text = ""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            text += "".join(b.text for b in message.content if isinstance(b, TextBlock))
    return text


show("🤖 Claude Agent SDK", await ask_agent(task_prompt), "magenta")

### 1.5.2 Which should you use? <a id="s1-5-2"></a>

**Result:** both answered the same question, but the Agent SDK needed more code, because it is built for **multi-step, tool-using agents**.

| Use | When you are saying |
|---|---|
| **Messages API** | *"I have one prompt and want one answer."* (1.1–1.4) |
| **Agent SDK** | *"Go through this folder of contracts, flag non-standard termination clauses, and write a summary file."* Claude decides what to do next |

## Key takeaways

### 01 · Tokens and the context window set the budget.
Tokens are what the model reads and what you pay for: about 1.3 tokens per English word, and more for code, JSON and non-English text. Input and output share one window (`input_tokens + max_tokens ≤ window`), so long documents need chunking or RAG. Put stable content first and changing content last, and never put timestamps or IDs at the front of the system prompt, or you lose the cache.

### 02 · Design around non-determinism.
Temperature trades consistency for variety: tune temperature **or** `top_p`, not both. Even at `temperature=0` answers are not perfectly repeatable, so test for correctness and properties, not exact strings.

### 03 · Choose the model and the thinking from your eval.
Start with Sonnet. Move to Opus only if it fails your eval, and to Haiku only if cost or latency demands it. Use an alias in development and pin the ID once you have a validated eval suite. Thinking tokens are output tokens, so turn extended or adaptive thinking on only when basic mode fails.

### 04 · Give the model only the guidance it needs.
Zero-, one- and few-shot differ only in the number of examples: add as many as the task needs and cover the hard cases. The API is stateless, so a conversation means resending the history, and the bill grows with it.

### 05 · Match the calling pattern to who is waiting.
Use the SDK in production for retries, streaming and types. Sync for scripts, streaming for anything a user watches, async for several answers at once, and batch for bulk work that can wait (within 24 hours, 50% cheaper). Use the Messages API for one answer and the Agent SDK for a multi-step, tool-using agent.